📌 Задача №1: Простые расчеты с RDD
Описание:

Создать простой массив чисел и рассчитать среднее арифметическое всех элементов массива с помощью RDD.

### Задание

Используя библиотеку PySpark, вычислите среднее арифметическое значение заданного набора чисел.

#### Шаги задания:

1. Создать объект `SparkSession` с именем приложения `"SimpleAvg"`.
2. Создать распределенный набор данных (`RDD`) с использованием метода `.parallelize()` из списка целых чисел `[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]`.
3. Рассчитать сумму всех элементов набора с помощью операции `.reduce()`, применяя лямбду-функцию сложения.
4. Определить количество элементов в наборе с помощью метода `.count()`.
5. Найти среднее арифметическое путем деления общей суммы на количество элементов.
6. Вывести итоговое среднее значение.

Результат выполнения программы должен выглядеть следующим образом:
```
Среднее значение: 5.5
```


Задание:

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("SimpleAvg").getOrCreate()
sc = spark.sparkContext

# Создание RDD с синтетическим набором чисел
numbers = sc.parallelize([1, 5, 8, 9, 10, 11, 12, 13, 14, 16, 19, 20, 21, 23, 24, 26, 27, 29, 36, 40])

# Расчет среднего значения
avg_value = numbers.reduce(lambda a, b: a + b) / numbers.count()

print(f"Ср. знач.: {avg_value}")


Ср. знач.: 18.2


📌 Задача №2: Анализ рейтинга фильмов с DataFrame API
Описание:

Имеется файл с рейтингами фильмов (movie_ratings.csv). Нужно выбрать фильмы с рейтингом выше 4 и посчитать количество просмотров для каждого жанра.
Данные:
movie_id	genre	rating
- 1	Drama	4.5
- 2	Comedy	3.8
- 3	Action	4.2
- 4	Sci-Fi	4.8
- 5	Romance	3.9

### Задание

Необходимо проанализировать рейтинги фильмов, используя библиотеку PySpark. Для этого нужно выполнить следующие шаги:

1. **Создать объект SparkSession** с названием приложения `"MovieRatings"`.
   
2. **Генерация DataFrame**: создать датасет на основе массива объектов Python, представляющих фильмы с полями:
   - `movie_id`: уникальный идентификатор фильма,
   - `genre`: жанр фильма ("Drama", "Comedy", "Action", "Sci-Fi", "Romance"),
   - `rating`: рейтинг фильма (число с плавающей точкой).

3. **Фильтрация**: выбрать только те фильмы, чей рейтинг превышает 4 балла.

4. **Агрегация**: сгруппировать отобранные фильмы по жанру и посчитать количество фильмов каждого жанра.

5. **Вывести результат**: показать итоговую таблицу с подсчитанным количеством высоко оцененных фильмов по каждому жанру.

---

Пример результата выполнения программы:

```
+-------+--------+
| genre | view_count|
+-------+--------+
|Drama  |       1 |
|Sci-Fi |       1 |
|Action |       1 |
+-------+--------+
```

Задание:

## Примечание

Исходя из текста и описания задания не очень понятно, нужно ли загружать внешний файл или нужно создавать объект внутри кода, поэтому ниже будет 2 версии кода, одна для работы с объектом, созданным внутри кода, другая - для загрузки внешнего файла.

## Версия с созданием объекта с данными внутри кода

In [16]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count

spark = SparkSession.builder.appName("MovieRatings").getOrCreate()

# Генерация синтетического фрейма данных
data = [
    {"movie_id": 1, "genre": "Drama", "rating": 4.5},
    {"movie_id": 2, "genre": "Comedy", "rating": 3.8},
    {"movie_id": 3, "genre": "Action", "rating": 4.2},
    {"movie_id": 4, "genre": "Sci-Fi", "rating": 4.8},
    {"movie_id": 5, "genre": "Romance", "rating": 3.9},
    {"movie_id": 6, "genre": "Drama", "rating": 3.5},
    {"movie_id": 7, "genre": "Comedy", "rating": 2.8},
    {"movie_id": 8, "genre": "Action", "rating": 3.2},
    {"movie_id": 9, "genre": "Sci-Fi", "rating": 3.8},
    {"movie_id": 10, "genre": "Romance", "rating": 5.0},
    {"movie_id": 11, "genre": "Drama", "rating": 4.0},
    {"movie_id": 12, "genre": "Comedy", "rating": 1.1},
    {"movie_id": 13, "genre": "Action", "rating": 4.8},
    {"movie_id": 14, "genre": "Sci-Fi", "rating": 4.1},
    {"movie_id": 15, "genre": "Romance", "rating": 1.9},
    {"movie_id": 16, "genre": "Drama", "rating": 1.5},
    {"movie_id": 17, "genre": "Comedy", "rating": 5.0},
    {"movie_id": 18, "genre": "Action", "rating": 1.2},
    {"movie_id": 19, "genre": "Sci-Fi", "rating": 1.8},
    {"movie_id": 20, "genre": "Romance", "rating": 5.0}
]

df = spark.createDataFrame(data)

# Выбор фильмов с высоким рейтингом и группировка по жанрам
rating_threshold = 4
high_rating_movies = df.filter(col("rating") > rating_threshold)
movies_grouped = high_rating_movies.groupBy("genre");
result = movies_grouped.agg(count("*").alias("movies_count"))

result.show()


+-------+------------+
|  genre|movies_count|
+-------+------------+
|Romance|           2|
|  Drama|           1|
| Action|           2|
| Sci-Fi|           2|
| Comedy|           1|
+-------+------------+



## Версия с загрузкой внешнего файла

In [19]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count
import os
import gdown

spark = SparkSession.builder.appName("MovieRatings").getOrCreate()

file_id = "17oGswcW0XXDuhVPIlvMIa0CXvv8FkzBh"
output = "movie_ratings.csv"

# Скачивание файла (если он не был скачан до этого)
if not os.path.exists(output):
  gdown.download(f"https://drive.google.com/uc?id={file_id}", output, quiet=False)

# Загрузка CSV во фрейм данных
df = spark.read.csv(output, header=True, inferSchema=True)

# Выбор фильмов с высоким рейтингом и группировка по жанрам
rating_threshold = 4
high_rating_movies = df.filter(col("rating") > rating_threshold)
movies_grouped = high_rating_movies.groupBy("genre");
result = movies_grouped.agg(count("*").alias("movies_count"))

result.show()

+-------+------------+
|  genre|movies_count|
+-------+------------+
|Romance|           2|
|  Drama|           1|
| Comedy|           1|
| Action|           2|
| Sci-Fi|           2|
+-------+------------+



-----------------------------------------------------------------

# Задача №3: Базовая обработка большого файла CSV
Описание:

Обработайте большой файл sales_data.csv, содержащий информацию о продажах магазинов. Посчитайте общую выручку магазина и выведите среднюю цену покупки.
Структура файла:
transaction_id	store_name	purchase_amount
- 1	Store A	100
- 2	Store B	150
- 3	Store C	200

### Задача

Используя библиотеку PySpark, выполните анализ продаж товаров разных магазинов. Необходимо реализовать следующую последовательность действий:

1. **Создание объекта SparkSession** с названием приложения `"SalesAnalysis"`.

2. **Формирование фрейма данных**: создайте DataFrame, содержащий информацию о магазинах и суммах покупок. Структура данных должна включать два поля:
   - `store_name`: название магазина («Store A», «Store B», «Store C»).
   - `purchase_amount`: сумма покупки (целое число).

3. **Расчёт суммарной выручки**: найдите общую сумму всех покупок во всех магазинах.

4. **Вычисление средней стоимости покупки**: определите среднюю стоимость одной покупки среди всех записей.

5. **Отображение результатов**: выведите две таблицы — одну с общей выручкой, вторую с вычисленным средним значением покупки.

---

Ожидаемый вывод:

```
+------+
|total|
+------+
| 970  |
+------+

+------------------+
| average_purchase |
+------------------+
|              161 |
+------------------+
```

Задание:

## Примечание

Исходя из текста и описания задания не очень понятно, нужно ли загружать внешний файл или нужно создавать объект внутри кода, поэтому ниже будет 2 версии кода, одна для работы с объектом, созданным внутри кода, другая - для загрузки внешнего файла.

## Версия с созданием объекта с данными внутри кода

In [24]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import mean
from pyspark.sql.functions import sum as spark_sum

spark = SparkSession.builder.appName("SalesAnalysis").getOrCreate()

# Создание синтетического фрейма данных
data = [
  ("Store A", 100),
  ("Store B", 150),
  ("Store C", 200),
  ("Store A", 120),
  ("Store B", 180),
  ("Store C", 220),
  ("Store A", 95),
  ("Store B", 310),
  ("Store C", 175),
  ("Store A", 260),
  ("Store B", 140),
  ("Store C", 390),
  ("Store A", 85),
  ("Store B", 230),
  ("Store C", 115),
  ("Store A", 445),
  ("Store B", 190),
  ("Store C", 275),
  ("Store A", 330),
  ("Store B", 160),
  ("Store C", 410),
  ("Store A", 205),
  ("Store B", 350),
  ("Store C", 130),
  ("Store A", 480),
  ("Store B", 245),
  ("Store C", 320),
  ("Store A", 170),
  ("Store B", 290),
  ("Store C", 465)
]

columns = ["store_name", "purchase_amount"]
df = spark.createDataFrame(data, columns)

# Общая выручка и средняя цена покупок
total_revenue = df.agg(spark_sum("purchase_amount").alias("total"))
average_price = df.agg({"purchase_amount": "avg"}).withColumnRenamed("avg(purchase_amount)", "average_purchase")

total_revenue.show()
average_price.show()


+-----+
|total|
+-----+
| 7235|
+-----+

+------------------+
|  average_purchase|
+------------------+
|241.16666666666666|
+------------------+



## Версия с загрузкой внешнего файла

In [25]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import mean
from pyspark.sql.functions import sum as spark_sum

spark = SparkSession.builder.appName("SalesAnalysis").getOrCreate()

file_id = "1lCaFvPmQkQsL4Z5OQTC3CwbQ2XAKV4ga"
output = "sales_data.csv"

# Скачивание файла (если он не был скачан до этого)
if not os.path.exists(output):
  gdown.download(f"https://drive.google.com/uc?id={file_id}", output, quiet=False)

# Загрузка CSV во фрейм данных
df = spark.read.csv(output, header=True, inferSchema=True)

columns = ["store_name", "purchase_amount"]
df = spark.createDataFrame(data, columns)

# Общая выручка и средняя цена покупок
total_revenue = df.agg(spark_sum("purchase_amount").alias("total"))
average_price = df.agg({"purchase_amount": "avg"}).withColumnRenamed("avg(purchase_amount)", "average_purchase")

total_revenue.show()
average_price.show()


+-----+
|total|
+-----+
| 7235|
+-----+

+------------------+
|  average_purchase|
+------------------+
|241.16666666666666|
+------------------+



---------------------------------------------------------------------

# Задача №4: Машинное обучение с помощью MLlib
Описание:

Реализовать простую классификацию объектов по двум признакам (рост и вес) с использованием логистической регрессии. Синтетические данные содержат признаки и метки классов.
Данные:
- height	weight	class_label
- 170	    65	    0
- 180	75	1
- 160	55	0

## Задание

Реализовать классификацию данных методом логистической регрессии с использованием библиотеки PySpark MLlib. Выполнить следующий алгоритм:

1. **Инициализация Spark Session:** Создайте экземпляр класса `SparkSession` с именем приложения `"LogRegClassification"`.

2. **Подготовка тренировочных данных:** Подготовьте синтетический набор данных, состоящий из четырёх строк с признаками роста (`height`), веса (`weight`) и меткой класса (`label`). Метка класса принимает значения `0` или `1`. Данные представлены в виде спискового формата, содержащего объекты типа `Row`.

3. **Преобразование данных:** Преобразуйте исходные признаки в числовой вектор, совместимый с моделью логистической регрессии. Используйте метод преобразования данных в вектор типа `Vectors.dense`.

4. **Обучение модели:** Обучите модель логистической регрессии, задав максимальные итерации равными `10` и регуляризационный параметр равным `0.01`.

5. **Предсказания:** Примените построенную модель к преобразованным данным и получите предсказанные вероятности принадлежности классов и фактические прогнозы.

6. **Показать результат:** Отобразите полученный фрейм данных с результатами классификации.

---

Пример вывода программы:

```
+-----+------+------------+----------+-------------------+--------------------+
|height|weight|     features|    label|           rawPrediction|            probability|
+-----+------+------------+----------+-------------------+--------------------+
|  170|    65|[170.0,65.0]|      0.0|(2,[0],[-0.3...])|[0.425557483148165,...|
|  180|    75|[180.0,75.0]|      1.0|(2,[1],[0.3...])|[0.5744425168518346,...|
|  160|    55|[160.0,55.0]|      0.0|(2,[0],[-0.5...])|[0.3775406687981454,...|
|  190|    80|[190.0,80.0]|      1.0|(2,[1],[0.5...])|[0.6224593312018546,...|
+-----+------+------------+----------+-------------------+--------------------+
```

Задание:

In [30]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.linalg import Vectors
from pyspark.sql import Row

spark = SparkSession.builder.appName("LogRegClassification").getOrCreate()

# Синтезированные данные
training_data = [
  Row(height=170, weight=65, label=0),
  Row(height=180, weight=75, label=1),
  Row(height=160, weight=55, label=0),
  Row(height=190, weight=80, label=1),
  Row(height=165, weight=60, label=0),
  Row(height=175, weight=70, label=1),
  Row(height=155, weight=50, label=0),
  Row(height=185, weight=78, label=1),
  Row(height=168, weight=63, label=0),
  Row(height=182, weight=77, label=1),
  Row(height=158, weight=52, label=0),
  Row(height=192, weight=83, label=1),
  Row(height=163, weight=58, label=0),
  Row(height=178, weight=72, label=1),
  Row(height=153, weight=48, label=0),
  Row(height=188, weight=82, label=1),
  Row(height=167, weight=62, label=0),
  Row(height=177, weight=74, label=1),
  Row(height=157, weight=54, label=0),
  Row(height=187, weight=79, label=1)
]

schema = ['height', 'weight', 'label']
df = spark.createDataFrame(training_data, schema)

# Конвертирование признаков в вектор
def transform_to_vector(source_row):
    return source_row.height, source_row.weight, Vectors.dense(float(source_row.height), float(source_row.weight)), source_row.label

df_to_map = df.rdd.map(transform_to_vector)
transformed_df = df_to_map.toDF(['height', 'weight', 'features', 'label'])

# Логистическая регрессия
max_iter = 10
reg_param = 0.01
lr = LogisticRegression(maxIter=max_iter, regParam=reg_param)
model = lr.fit(transformed_df)

predictions = model.transform(transformed_df)
predictions.show()


+------+------+------------+-----+--------------------+--------------------+----------+
|height|weight|    features|label|       rawPrediction|         probability|prediction|
+------+------+------------+-----+--------------------+--------------------+----------+
|   170|    65|[170.0,65.0]|    0|[0.98686741282360...|[0.72846873388116...|       0.0|
|   180|    75|[180.0,75.0]|    1|[-2.9015584606919...|[0.05207657655277...|       1.0|
|   160|    55|[160.0,55.0]|    0|[4.87529328633915...|[0.99242496387609...|       0.0|
|   190|    80|[190.0,80.0]|    1|[-5.7346647678747...|[0.00322155369460...|       1.0|
|   165|    60|[165.0,60.0]|    0|[2.93108034958137...|[0.94936163714400...|       0.0|
|   175|    70|[175.0,70.0]|    1|[-0.9573455239341...|[0.27740997985659...|       1.0|
|   155|    50|[155.0,50.0]|    0|[6.81950622309692...|[0.99890893160852...|       0.0|
|   185|    78|[185.0,78.0]|    1|[-4.4236435709166...|[0.01184839670990...|       1.0|
|   168|    63|[168.0,63.0]|    